# Torch-KWT Tutorial

This notebook walks through downloading Google Speech Commands V2, running inference, and training a KWT model with the current Torch-KWT codebase.


## Setup

### 1. Clone the repository

In [ ]:
!git clone https://github.com/PositiveLoss/kwt.git

In [ ]:
%cd kwt

### 2. Install dependencies

The project is managed with `uv`. Shell commands in this notebook use `uv run` so they execute inside the project environment.


In [ ]:
!pip install -q uv
!uv sync

### 3. Download the Google Speech Commands V2 dataset

We'll be saving it to the `./data/` folder.

In [ ]:
!sh ./download_gspeech_v2.sh ./data/

In [ ]:
!ls data

The dataset provides `validation_list.txt` and `testing_list.txt`. `train.py` can generate `training_list.txt` and `label_map.json` automatically if they are missing. You can also create them manually with `make_data_list.py`:


In [ ]:
!uv run python make_data_list.py -v ./data/validation_list.txt -t ./data/testing_list.txt -d ./data/ -o ./data/

## Using Pre-trained Models for Inference

The original Torch-KWT pretrained KWT-1 checkpoint is a legacy PyTorch `.pth` file. The current code can still load it for compatibility. New training runs save `best.safetensors` and `last.safetensors`.


### 4. Download the Pretrained KWT-1 Model


In [ ]:
!wget -O "kwt1_pretrained.pth" "https://drive.google.com/uc?id=1y91PsZrnBXlmVmcDi26lDnpl4PoC5tXi&export=download"

### 5. Explore Some Test Set Audio Files

Let's take a few files from the test set, visualize the same repo feature extractor used by training/inference, and listen to them.


In [ ]:
!cat data/testing_list.txt | head -n 3

In [ ]:
from IPython.display import Audio
import matplotlib.pyplot as plt

from utils.audio import load_audio
from utils.dataset import extract_features

samples = [
    "./data/visual/3d86b69a_nohash_0.wav",
    "./data/dog/881583a6_nohash_0.wav",
    "./data/zero/4fd1443e_nohash_2.wav",
]

feature_settings = {
    "feature_type": "mfcc",
    "feature_time_bins": 98,
    "sr": 16000,
    "n_mels": 40,
    "n_fft": 480,
    "win_length": 480,
    "hop_length": 160,
    "coch_env_sr": 100,
    "coch_filter_n": 38,
    "coch_low_freq": 50.0,
    "coch_high_freq": 8000.0,
}


def show_features(audio_path, audio_settings=feature_settings):
    print(f"Showing {audio_path}")
    x = load_audio(audio_path, sr=audio_settings["sr"])
    features = extract_features(x, audio_settings)

    plt.figure(figsize=(7, 5))
    plt.imshow(features, cmap="hot", aspect="auto", origin="lower")
    plt.title(audio_settings["feature_type"])
    plt.xlabel("time")
    plt.ylabel("feature bin")
    plt.colorbar()
    plt.show()

In [ ]:
show_features(samples[0])
Audio(samples[0])

In [ ]:
show_features(samples[1])
Audio(samples[1])

In [ ]:
show_features(samples[2])
Audio(samples[2])

### 6. Use Inference Scripts

There are two inference scripts provided in the repository:
- `inference.py`
- `window_inference.py`

We'll use the pretrained model downloaded earlier. The examples use `config.yaml`, which defaults to MFCC features.

**Using `inference.py`:**

For simple short clips that are about 1 second long, such as the audios in the Speech Commands dataset, use `inference.py`. It can run on a single clip or a folder containing several clips.


In [ ]:
!uv run python inference.py --conf config.yaml \
                     --ckpt kwt1_pretrained.pth \
                     --inp data/dog/881583a6_nohash_0.wav \
                     --out outputs/single_preds/ \
                     --lmap data/label_map.json \
                     --device cpu

Let us check the prediction output:

In [ ]:
!cat outputs/single_preds/preds.json

Seems like it's correct!

Similarly, you can also provide **a folder containing multiple audio wavs** to the `--inp` argument, instead of a single audio path. Let's take the last 3 audios in testing list and put them in a folder:

In [ ]:
# Moving multiple audios to a single folder "samples"
!cat data/testing_list.txt | tail -n 3
!mkdir -p samples/
!cp $(cat data/testing_list.txt | tail -n 3) samples/

In [ ]:
# Run inference
!uv run python inference.py --conf config.yaml \
                     --ckpt kwt1_pretrained.pth \
                     --inp samples/ \
                     --out outputs/folder_preds/ \
                     --lmap data/label_map.json \
                     --device cpu

The outputs are in a similar json format.

In [ ]:
!cat outputs/folder_preds/preds.json

*If you are running inference on a large number of clips, you can also use the **--batch_size** argument.* By default, batch size is set to 1 during inference.



---



---

**Using window_inference.py:**

In real life use-case, audios are typically much longer than 1s. To get predictions from long audios, we thus need to run multiple inferences on various "chunks" or "windows" in that audio clip.

![window.png](https://raw.githubusercontent.com/ID56/Torch-KWT/main/resources/window.png)

While we don't have a long clip here, we can try **concatenating the three wavs** inside the `samples/` folder we made above into **a single 3s long audio clip**.

In [ ]:
from IPython.display import Audio
import glob

import numpy as np
import soundfile as sf

from utils.audio import fix_length


def concat_audio(audio_folder):
    audio_list = []
    for audio_file in glob.glob(f"{audio_folder}/*.wav"):
        x = load_audio(audio_file, sr=16000)
        x = fix_length(x, 16000)
        audio_list.append(x)
    long_audio = np.hstack(audio_list)
    sf.write("long_audio.wav", long_audio, 16000)


concat_audio("samples/")
Audio("long_audio.wav")

Now let's run `window_inference.py`. It can be run in three modes:

- multi: saves all found predictions (default)
- max: saves the "most confident" prediction (outputs only a single 'clipwise; prediction for the whole clip)
- n_voting: saves the "most frequent" prediction (outputs only a single 'clipwise' prediction for the whole clip)

The `multi` mode is probably the most useful, and is thus the default mode. We will run inferences over **1s windows**, and **slide the window by 0.5s**. We will only keep the predictions with **confidence over 0.85**.

In [ ]:
!uv run python window_inference.py --conf config.yaml \
                     --ckpt kwt1_pretrained.pth \
                     --inp long_audio.wav \
                     --out outputs/long_audio/ \
                     --lmap data/label_map.json \
                     --device cpu \
                     --wlen 1 \
                     --stride 0.5 \
                     --thresh 0.85 \
                     --mode multi

In [ ]:
!cat outputs/long_audio/preds_clip.json

- Outputs are in the form: **[class, confidence, start_sample, end_sample]**. (If you divide sample by sample rate, you can get the time range)
- It is possible that sometimes two windows may both partially fall on the same keyword instance. In such a case, you may get two consecutive predictions of the same class with some overlap. *(Like, nine-1s-2s and nine-1.5s-2.5s.)*
- Similar to `inference.py` the `--inp` argument in `window_inference.py` can also work with a folder of audio files
- You can also set the --batch_size argument for faster inference on GPU

## Training

For training, we only need to provide the config file.

### 9. Set Up a Config File

For this demo, we'll write a short 10-epoch config under `configs/kwt1_demo.yaml`. The top-level `config.yaml` is the full default MFCC config. You can also train with `config_cochleagram.yaml` or `config_connear.yaml`.

Metric logging is handled by local stdout/file logging by default. Set `exp.trackio: True` if you want Trackio logging.


In [ ]:
conf_str = """# sample config to run a demo training of 10 epochs

data_root: ./data/
train_list_file: ./data/training_list.txt
val_list_file: ./data/validation_list.txt
test_list_file: ./data/testing_list.txt
label_map: ./data/label_map.json

exp:
    trackio: False
    trackio_space_id:
    trackio_server_url:
    proj_name: torch-kwt-demo
    exp_dir: ./runs
    exp_name: exp-demo-0.0.1
    device: auto
    log_freq: 20
    log_to_file: True
    log_to_stdout: True
    warm_cache: False
    val_freq: 1
    n_workers: 1
    pin_memory: True
    cache: 2
    feature_cache: False
    feature_cache_dir: ./data/.feature_cache_demo

hparams:
    seed: 0
    batch_size: 512
    grad_accum_steps: 1
    n_epochs: 10
    l_smooth: 0.1

    audio:
        feature_type: mfcc
        feature_time_bins: 98
        sr: 16000
        n_mels: 40
        n_fft: 480
        win_length: 480
        hop_length: 160
        coch_env_sr: 100
        coch_filter_n: 38
        coch_low_freq: 50.0
        coch_high_freq: 8000.0
        connear_weights_path: ./data/connear/Gmodel.pt
        connear_auto_download: False
        connear_device: auto
        connear_sr: 20000
        connear_batch_size: 4
        connear_n_channels: 201
        connear_log_scale: 1000000.0
        connear_normalize: True
        connear_input_scale: 1.0

    model:
        name:
        input_res: [201, 98]
        patch_res: [201, 1]
        num_classes: 35
        mlp_dim: 256
        dim: 64
        heads: 1
        depth: 12
        dropout: 0.0
        emb_dropout: 0.1
        pre_norm: False
        use_sdpa: True
        use_helion_kernels: False

    optimizer:
        opt_type: adamw
        opt_kwargs:
          lr: 0.001
          weight_decay: 0.1

    scheduler:
        n_warmup: 1
        max_epochs: 10
        scheduler_type: cosine_annealing

    augment:
        spec_aug:
            n_time_masks: 2
            time_mask_width: 25
            n_freq_masks: 2
            freq_mask_width: 7
"""

!mkdir -p configs
with open("configs/kwt1_demo.yaml", "w+") as f:
    f.write(conf_str)

### 10. Initiate Training

Make sure you are using a GPU runtime for real training. With `exp.cache: 2`, the dataset caches extracted features in memory, so waveform-level augmentations such as resampling, time shift, and background noise are skipped. Spectral augmentation still runs during training.

New checkpoints are saved as safetensors: `runs/<exp_name>/best.safetensors` and `runs/<exp_name>/last.safetensors`.


Feature options:

- `feature_type: mfcc` uses spafe-rs MFCC features.
- `feature_type: cochleagram` uses spafe-rs cochleagram features. Use `config_cochleagram.yaml` for a ready config.
- `feature_type: connear` uses the pretrained CoNNear cochlea model. First run `uv run python download_connear_model.py`, then train with `config_connear.yaml`; that config keeps all 201 CoNNear channels.


In [ ]:
!uv run python train.py --conf configs/kwt1_demo.yaml

After training, inspect `runs/exp-demo-0.0.1/training_log.txt` and the generated safetensors checkpoints. For a full reproduction-style run, use the full default `config.yaml` and set `hparams.n_epochs`/scheduler settings as needed.
